In [12]:
from constants import *
print(CORES, FREQ_LEVELS)

4 [0.4, 0.6, 0.8, 1.0]


In [13]:
from tasks import generate_tasks

tasks = generate_tasks()
for t in tasks[:10]:
    print(t)

T0(rem=7, dl=175)
T1(rem=7, dl=68)
T2(rem=5, dl=118)
T3(rem=4, dl=155)
T4(rem=17, dl=208)
T5(rem=9, dl=37)
T6(rem=10, dl=78)
T7(rem=7, dl=120)
T8(rem=18, dl=51)
T9(rem=6, dl=124)


In [14]:
from dvfs import get_frequency

print(get_frequency(0.55))  # expect 0.6
print(get_frequency(0.9))   # expect 1.0

0.6
1.0


In [15]:
from tasks import generate_tasks
from scheduler import Scheduler

tasks = generate_tasks()
sched = Scheduler(tasks)

for _ in range(50):
    sched.step()

    print(f"\nTick {sched.time}")
    
    # Core status
    for core in sched.cores:
        if core.current_task:
            print(f"Core {core.cid} → T{core.current_task.tid} @ {core.freq} GHz")
        else:
            print(f"Core {core.cid} → Idle")
    
    # Queue info
    print("Queue size:", len(sched.ready_queue))
    print("Queue:", [t.tid for t in sched.ready_queue])

Core temps: [(0, 35.0), (1, 35.0), (2, 35.0), (3, 35.0)]

Tick 1
Core 0 → Idle
Core 1 → Idle
Core 2 → Idle
Core 3 → Idle
Queue size: 0
Queue: []
Core temps: [(0, 35.0), (1, 35.0), (2, 35.0), (3, 35.0)]

Tick 2
Core 0 → T5 @ 0.6 GHz
Core 1 → Idle
Core 2 → Idle
Core 3 → Idle
Queue size: 0
Queue: []
Core temps: [(0, 35.5), (1, 35.0), (2, 35.0), (3, 35.0)]

Tick 3
Core 0 → T5 @ 0.6 GHz
Core 1 → T26 @ 1.0 GHz
Core 2 → Idle
Core 3 → Idle
Queue size: 0
Queue: []
Core temps: [(0, 36.0), (1, 35.5), (2, 35.0), (3, 35.0)]

Tick 4
Core 0 → T5 @ 0.6 GHz
Core 1 → T26 @ 1.0 GHz
Core 2 → Idle
Core 3 → Idle
Queue size: 0
Queue: []
Core temps: [(0, 36.5), (1, 36.0), (2, 35.0), (3, 35.0)]

Tick 5
Core 0 → T5 @ 0.6 GHz
Core 1 → T26 @ 1.0 GHz
Core 2 → T32 @ 0.6 GHz
Core 3 → Idle
Queue size: 0
Queue: []
Core temps: [(0, 37.0), (1, 36.5), (2, 35.5), (3, 35.0)]

Tick 6
Core 0 → T5 @ 0.6 GHz
Core 1 → T26 @ 1.0 GHz
Core 2 → T32 @ 0.6 GHz
Core 3 → Idle
Queue size: 0
Queue: []
Core temps: [(0, 37.5), (1, 37.0), (

In [16]:
print(f"T5 util:", [t.utilization for t in tasks if t.tid == 5])
print(f"T26 util:", [t.utilization for t in tasks if t.tid == 26])

T5 util: [0.54]
T26 util: [0.83]


In [17]:
completed = sum(1 for t in tasks if t.remaining <= 0)
print("Completed:", completed)

Completed: 10


In [18]:
for _ in range(10):
    sched.step()

    print(f"\nTick {sched.time}")

    for core in sched.cores:
        if core.current_task:
            print(f"Core {core.cid} → T{core.current_task.tid} @ {core.freq} GHz | Temp: {core.temperature:.1f}")
        else:
            print(f"Core {core.cid} → Idle | Temp: {core.temperature:.1f}")

    print("Queue size:", len(sched.ready_queue))

Core temps: [(0, 58.1), (1, 55.5), (2, 55.9), (3, 53.1)]

Tick 51
Core 0 → T49 @ 0.4 GHz | Temp: 58.6
Core 1 → T1 @ 0.4 GHz | Temp: 56.0
Core 2 → T48 @ 0.8 GHz | Temp: 56.4
Core 3 → T52 @ 0.6 GHz | Temp: 53.6
Queue size: 6
Core temps: [(0, 58.6), (1, 56.0), (2, 56.4), (3, 53.6)]

Tick 52
Core 0 → T49 @ 0.4 GHz | Temp: 59.1
Core 1 → T1 @ 0.4 GHz | Temp: 56.5
Core 2 → T48 @ 0.8 GHz | Temp: 56.9
Core 3 → T52 @ 0.6 GHz | Temp: 54.1
Queue size: 6
Core temps: [(0, 59.1), (1, 56.5), (2, 56.9), (3, 54.1)]

Tick 53
Core 0 → T49 @ 0.4 GHz | Temp: 59.6
Core 1 → T1 @ 0.4 GHz | Temp: 57.0
Core 2 → T48 @ 0.8 GHz | Temp: 57.4
Core 3 → T52 @ 0.6 GHz | Temp: 54.6
Queue size: 6
Core temps: [(0, 59.6), (1, 57.0), (2, 57.4), (3, 54.6)]

Tick 54
Core 0 → T49 @ 0.4 GHz | Temp: 60.1
Core 1 → T1 @ 0.4 GHz | Temp: 57.5
Core 2 → T48 @ 0.8 GHz | Temp: 57.9
Core 3 → T52 @ 0.6 GHz | Temp: 55.1
Queue size: 6
Core temps: [(0, 60.1), (1, 57.5), (2, 57.9), (3, 55.1)]

Tick 55
Core 0 → Idle | Temp: 59.9
Core 1 → T1 @ 0

In [19]:
from power_model import calculate_power

print(calculate_power(sched.cores[0]))

0.05


In [20]:
from thermal import update_temperature

core = sched.cores[0]
update_temperature(core, 1.5)
print(core.temperature)

59.810999999999986


In [21]:
for _ in range(60):
    sched.step()

    print(f"\nTick {sched.time}")
    for core in sched.cores:
        if core.current_task:
            print(f"Core {core.cid} → T{core.current_task.tid} @ {core.freq} GHz | Temp: {core.temperature:.1f}")
        else:
            print(f"Core {core.cid} → Idle | Temp: {core.temperature:.1f}")

    print("Queue size:", len(sched.ready_queue))

Core temps: [(0, 59.8), (1, 59.8), (2, 60.2), (3, 58.1)]

Tick 61
Core 0 → Idle | Temp: 59.6
Core 1 → T44 @ 0.6 GHz | Temp: 60.3
Core 2 → Idle | Temp: 60.0
Core 3 → T52 @ 0.6 GHz | Temp: 58.6
Queue size: 8
Core temps: [(0, 59.6), (1, 60.3), (2, 60.0), (3, 58.6)]

Tick 62
Core 0 → T49 @ 0.4 GHz | Temp: 60.1
Core 1 → Idle | Temp: 60.1
Core 2 → T29 @ 1.0 GHz | Temp: 60.5
Core 3 → T52 @ 0.6 GHz | Temp: 59.1
Queue size: 8
Core temps: [(0, 60.1), (1, 60.1), (2, 60.5), (3, 59.1)]

Tick 63
Core 0 → Idle | Temp: 59.9
Core 1 → Idle | Temp: 59.9
Core 2 → Idle | Temp: 60.3
Core 3 → T52 @ 0.6 GHz | Temp: 59.6
Queue size: 11
Core temps: [(0, 59.9), (1, 59.9), (2, 60.3), (3, 59.6)]

Tick 64
Core 0 → T49 @ 0.4 GHz | Temp: 60.4
Core 1 → T44 @ 0.6 GHz | Temp: 60.4
Core 2 → Idle | Temp: 60.1
Core 3 → T52 @ 0.6 GHz | Temp: 60.1
Queue size: 10
Core temps: [(0, 60.4), (1, 60.4), (2, 60.1), (3, 60.1)]

Tick 65
Core 0 → Idle | Temp: 60.2
Core 1 → Idle | Temp: 60.2
Core 2 → Idle | Temp: 59.9
Core 3 → Idle | Te

In [22]:
completed = sum(1 for t in tasks if t.remaining <= 0)
print("Completed:", completed)

Completed: 12
